
# Data Description – Stack Overflow Thread Data (2021–2023)

This document describes the variables, data sources, collection methods, time coverage, data volume, and known limitations of the dataset generated by `AI+RU.ipynb`.

## Data Source

- **Original provider:** Stack Exchange Inc.
- **Public dump:** [Stack Exchange Data Dump](https://archive.org/details/stackexchange) (CC BY-SA 4.0 license)
- **Selected communities:**
  - `ai.stackexchange.com` (AI Stack Exchange)
  - `rus.stackexchange.com` (Russian Stack Exchange)

## Data Collection Method

The raw XML files (`Posts.xml`, `Comments.xml`, `Users.xml`) are parsed iteratively to avoid loading entire files into memory. Only records with `CreationDate` between **2021-01-01** and **2023-12-31** are retained. The extraction logic:

1. **Users** – Read all users to build a lookup dictionary (user ID → reputation).
2. **First pass over Posts** – Collect all questions in the date range and build a mapping from answer IDs to their parent question IDs.
3. **Comments** – Filter comments by date, link each comment to its ultimate question ID (via direct question or parent answer), and write to CSV.
4. **Questions** – Write all collected questions to CSV.
5. **Second pass over Posts** – Write all answers in the date range, marking whether each answer is accepted.
6. **Users** – Write only those users who appear in the filtered posts or comments.
7. **Samples** – Create small subsets (first 1000 questions + related content) for quick exploration.

## Time Coverage

- **Start date:** 2021-01-01
- **End date:** 2023-12-31
- The period includes the release of ChatGPT (November 2022), enabling pre‑/post‑analysis.

## Variables by CSV File

### `{platform}_questions_raw.csv`

| Column               | Type      | Description |
|----------------------|-----------|-------------|
| `question_id`        | integer   | Unique identifier of the question (PostId where PostTypeId=1) |
| `creation_date`      | string    | UTC timestamp when the question was posted (format: YYYY-MM-DD HH:MM:SS) |
| `body`               | string    | Raw HTML/XML content of the question |
| `tags`               | string    | Space‑separated tags enclosed in `<` `>` (e.g. `<python><ai>`) |
| `owner_user_id`      | integer   | User ID of the asker (may be NULL if user was deleted) |
| `score`              | integer   | Net votes (upvotes minus downvotes) |
| `accepted_answer_id` | integer   | ID of the answer marked as accepted (NULL if none) |
| `post_type_id`       | integer   | Always `1` (question) |

### `{platform}_answers_raw.csv`

| Column                 | Type      | Description |
|------------------------|-----------|-------------|
| `answer_id`            | integer   | Unique identifier of the answer (PostTypeId=2) |
| `parent_question_id`   | integer   | ID of the question this answer belongs to |
| `creation_date`        | string    | UTC timestamp of the answer |
| `body`                 | string    | Raw HTML/XML content of the answer |
| `owner_user_id`        | integer   | User ID of the answerer (may be NULL) |
| `score`                | integer   | Net votes |
| `is_accepted`          | integer   | `1` if this answer is the accepted answer for its parent question, otherwise `0` |

### `{platform}_comments_raw.csv`

| Column              | Type      | Description |
|---------------------|-----------|-------------|
| `comment_id`        | integer   | Unique identifier of the comment |
| `post_id`           | integer   | ID of the post (question or answer) that the comment is attached to |
| `user_id`           | integer   | User ID of the comment author (may be NULL) |
| `creation_date`     | string    | UTC timestamp of the comment |
| `text`              | string    | Content of the comment (plain text with minimal HTML) |
| `parent_comment_id` | integer   | **Always NULL** (threaded comments not extracted in this version) |
| `question_id`       | integer   | Inferred top‑level question ID (the ultimate question this comment belongs to, even if the comment is on an answer) |

### `{platform}_users_raw.csv`

| Column        | Type      | Description |
|---------------|-----------|-------------|
| `user_id`     | integer   | Unique user identifier |
| `reputation`  | integer   | User's reputation score at the time of the data dump (not time‑varying) |

## Data Volume (Approximate)

| Platform | Questions | Answers | Comments | Users (filtered) |
|----------|-----------|---------|----------|------------------|
| AI       | (varies)  | (varies) | (varies) | (varies)         |
| Russian  | (varies)  | (varies) | (varies) | (varies)         |

Actual counts depend on the specific data dump version and the date filter. For reference, the full AI Stack Exchange (up to 2023) typically contains ~10k–20k questions; the Russian site is smaller.

## Known Data Issues & Limitations

1. **Missing `parent_comment_id`** – The extraction script does not parse comment threading; therefore `parent_comment_id` is always `NULL`. All comments are treated as top‑level comments on their immediate post.

2. **Deleted users** – `owner_user_id` and `user_id` can be `NULL` when the user account was deleted. Such records are still included, but linkage to user reputation is impossible.

3. **User reputation is static** – The reputation value is taken from the dump’s `Users.xml` snapshot, not from the exact time of the post/comment. This may cause minor inaccuracies for longitudinal analyses.

4. **Sample datasets** – The `sample_*` files contain only the first 1000 questions (by order of appearance in the original XML) and their related answers/comments/users. These are **not** a random sample and should not be used for statistical inference.

5. **Date filtering** – The filter applies only to `CreationDate`. Deleted posts (marked `DeletionDate`) are not excluded; they may appear with NULL owners or other anomalies.

6. **XML parsing limitations** – Very long fields (e.g., `Body` with embedded HTML) are stored as raw strings, which may contain unescaped characters. However, they remain valid for text mining.

7. **No edit history** – Only the final version of each post/comment is included. Edits, rollbacks, or comment deletions are not captured.

8. **Comments linked to questions** – The `question_id` field in `comments_raw.csv` is inferred by first checking if `post_id` is a question; if not, it looks up the parent question of the answer that the comment is on. For comments on answers where the answer’s parent question is missing (corrupt data), the comment is dropped.

## Recommended Usage

- Use the **full** CSVs for quantitative analysis of commenting behavior, answer acceptance, and user activity.
- Use the **sample** CSVs only for code testing and exploratory visualization.
- Always filter out rows with missing `question_id` in comments if you require strict thread linkage.
- For time‑series analysis, parse `creation_date` as datetime and aggregate at daily/weekly levels.

## License

The original Stack Exchange data is provided under the [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license. This derived dataset follows the same license.